<a href="https://colab.research.google.com/github/duygusezr/BMI-Calculation-and-Zodiac-Finder/blob/main/masal_ozeti.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
!pip install transformers datasets accelerate

In [15]:
from google.colab import files
uploaded = files.upload()

Saving masal_dataset_guncel.csv to masal_dataset_guncel.csv


In [16]:
from datasets import load_dataset

dataset = load_dataset("csv", data_files="masal_dataset_guncel.csv", delimiter=";")

Generating train split: 0 examples [00:00, ? examples/s]

In [17]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "csebuetnlp/mT5_multilingual_XLSum"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


/usr/local/lib/python3.11/dist-packages/transformers/convert_slow_tokenizer.py:559: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [18]:
def preprocess_function(example):
    inputs = tokenizer(example["text"], truncation=True, padding="max_length", max_length=512)
    targets = tokenizer(example["summary"], truncation=True, padding="max_length", max_length=128)
    inputs["labels"] = targets["input_ids"]
    return inputs

tokenized_dataset = dataset.map(preprocess_function, batched=True)


Map:   0%|          | 0/7 [00:00<?, ? examples/s]

In [19]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

# Eğitim ayarları
training_args = Seq2SeqTrainingArguments(
    output_dir="./masal_model",
    learning_rate=2e-5,
    per_device_train_batch_size=1,  # Düşük GPU kullanımı
    num_train_epochs=10,
    save_strategy="epoch",
    logging_dir="./logs",
    fp16=True  # GPU varsa hız ve bellek avantajı
)

# Data collator (etiketleri otomatik hizalar)
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# Trainer tanımı
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    tokenizer=tokenizer,
    data_collator=data_collator
)


<ipython-input-19-5d8eb49acd3e>:18: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [ ]:
trainer.train()

Step,Training Loss


/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:3339: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 84, 'num_beams': 4, 'length_penalty': 0.6, 'no_repeat_ngram_size': 2}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


In [12]:
model.save_pretrained("./masal_model")
tokenizer.save_pretrained("./masal_model")

('./masal_model/tokenizer_config.json',
 './masal_model/special_tokens_map.json',
 './masal_model/spiece.model',
 './masal_model/added_tokens.json',
 './masal_model/tokenizer.json')

In [13]:
from transformers import pipeline

ozetleme_pipeline = pipeline("summarization", model="./masal_model", tokenizer="./masal_model")

# Örnek test
text = "Bir varmış, bir yokmuş. Geniş bir ormanın en ucunda, diğerlerinden biraz daha küçük bir serçe yaşarmış. Adı Minik'miş. Minik’in kanatları biraz zayıf olduğu için hiç uçamazmış. Diğer kuşlar gökyüzünde süzülürken, Minik onları izler, iç çekermiş.Bir gün ormanda büyük bir yarış duyurulmuş: En yükseğe uçan kuş, gökyüzünün kralı olacak!Herkes çok heyecanlıymış. Minik de gizlice bu yarışa katılmak istemiş ama kimse onunla alay etmesin diye sessiz kalmış.Geceleri uyumadan önce gökyüzünü hayal eder, bulutların arasında süzüldüğünü düşünürmüş. Her sabah biraz daha uzağa zıplamaya çalışmış, her gün biraz daha cesaretini toplamış. Uçamasa da, çabalıyormuş.Yarış günü gelmiş. Tüm kuşlar sırayla kanat çırpmış. Bazıları çok yükseğe çıkmış, bazıları erkenden yorulmuş. Minik ise sahneye çıkıp kanatlarını çırpmamış. Bunun yerine bir ağacın tepesine zıplamış, sonra bir dala, sonra bir başka dala…Tüm orman sessizleşmiş. Minik zıplaya zıplaya, azimle en yüksek dala ulaşmış. Oradaki en yaşlı baykuş, gözlüklerini düzeltip gülümsemiş:“Uçmak sadece kanatla olmaz,” demiş. “Yürekle olur. Bugünün kazananı Minik’tir!”O günden sonra Minik, uçmayı başaramasa da, azmiyle herkese ilham olmuş. Ve her gece, rüyasında özgürce gökyüzünde uçmaya devam etmiş…"
ozet = ozetleme_pipeline(text)[0]["summary_text"]
print("📌 Özet:", ozet)


Device set to use cuda:0


📌 Özet: Güneydoğu Anadolu’da bir orman varmış.
